# Expliquer un détecteur d'intrusion
**Projet de module Machine Learning pour la cyber-sécurité — Cahier des charges 1**
Binôme : Oussama Hamada, Aladine Fadhli — ING-5-J-SSIRF-D, TEK-UP

**Question centrale :** le détecteur apprend-il des comportements d'attaque, ou des artefacts de la collecte du jeu CIC-IDS2017 ?

| Section | Contenu |
|---|---|
| 0 | Environnement et versions |
| 1 | Données et nettoyage (sans fuite) |
| 2 | Échantillonnage, découpage 60/20/20, sélection des variables sur l'entraînement |
| 3 | Modèles : baseline, forêt aléatoire, XGBoost (validation croisée) |
| 4 | Évaluation sur le test (une seule fois) et coût FP / FN |
| 5 | Tâche 2 : trois mesures d'importance globale |
| 6 | Tâche 3 : cinq alertes expliquées avec SHAP |
| 7 | Tâche 4 : audit de plausibilité du top 10 |
| 8 | Tâche 5 : ablation des variables suspectes |
| 9 | Tâche 6 : stabilité des explications |
| 10 | Extension : RF vs XGBoost, familles d'attaques |
| 11 | Conclusion et limites |

## 0. Environnement

In [ ]:
# --- Colab only: clone the repo and install the two libraries Colab lacks -------
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/AladinFadhli/ml-ids-explain.git"   # <-- your repo URL
    if not os.path.exists("ml-ids-explain"):
        !git clone -q $REPO_URL
    %cd ml-ids-explain
    !pip -q install shap xgboost
    # Option: data on Google Drive
    # from google.colab import drive; drive.mount('/content/drive')
    # os.environ["IDS_DATA_DIR"] = "/content/drive/MyDrive/cicids2017"

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings; warnings.filterwarnings("ignore")
import platform, numpy as np, pandas as pd, sklearn, xgboost, shap, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_colwidth", 120)

from src import config as C, data as D, models as M, evaluate as E, explain as X
np.random.seed(C.SEED)
C.FIG_DIR.mkdir(parents=True, exist_ok=True)

versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__, "shap": shap.__version__,
            "matplotlib": matplotlib.__version__, "seed": C.SEED, "data_dir": str(C.DATA_DIR)}
pd.Series(versions, name="version").to_frame()

## 1. Données et nettoyage

**Jeu :** CIC-IDS2017 (Canadian Institute for Cybersecurity) — flux réseau extraits par CICFlowMeter sur cinq jours de capture, étiquetés par type d'attaque. Fichiers CSV dans `data/raw/` (voir `data/README.md`).

**Défauts documentés** (Engelen, Rimmer, Joosen, *Troubleshooting an Intrusion Detection Dataset: the CICIDS2017 Case Study*, IEEE SPW 2021) : CICFlowMeter termine un flux TCP au premier FIN, ce qui crée des « appendices TCP » — plus de 25 % des flux sont des artefacts, jusqu'à 50 % pour certaines catégories — et plusieurs flux Web/Bot sont étiquetés comme attaques sans contenir de charge utile. Ces défauts justifient le nettoyage ci-dessous et l'audit de la section 7.

**Nettoyage sans opération apprise** (appliqué avant le découpage, car il ne dépend d'aucune statistique des données) :
1. suppression des colonnes identifiantes (Flow ID, IP, port source, horodatage) : elles permettent de « tricher » ;
2. suppression des lignes avec valeurs infinies ou manquantes ;
3. suppression des doublons exacts **avant** le découpage (sinon le même flux tombe à la fois en entraînement et en test) ;
4. suppression des flux aux variables identiques mais aux étiquettes différentes (étiquetage contradictoire).

Le *port de destination* est conservé dans le premier modèle mais marqué comme suspect : il nomme un service, pas un comportement (testé en section 8).

In [ ]:
raw = D.load_raw()
print(f"{len(raw):,} rows, {raw.shape[1]} columns from {raw['source_file'].nunique()} files")
raw.groupby("source_file")[C.LABEL_COL].value_counts().rename("flows").to_frame()

In [ ]:
log = []
df = D.clean(raw, log)
del raw
display(pd.DataFrame(log, columns=["step", "value"]))
fam_counts = df["family"].value_counts().rename("flows").to_frame()
fam_counts["share"] = fam_counts["flows"] / len(df)
fam_counts

## 2. Échantillonnage et découpage

- **Échantillon** : au plus `MAX_ROWS` flux, tirés proportionnellement par famille d'attaque ; les familles rares (moins de `KEEP_ALL_BELOW` flux) sont gardées entières. Le même échantillon sert à tous les modèles. Raison : calcul sur processeur (Colab gratuit).
- **Découpage 60 / 20 / 20 stratifié par famille, graine 42.** Pas de découpage par jour : chaque journée de CIC-IDS2017 contient des attaques différentes, un découpage par jour testerait des attaques jamais vues (c'est le sujet 4, dérive). La section 9 vérifie que les explications tiennent avec d'autres découpages.
- **Sélection apprise sur l'entraînement seulement** : colonnes constantes et colonnes dupliquées.
- **Contrôle de fuite** : aucune ligne de validation ou de test ne doit avoir le même vecteur de variables qu'une ligne d'entraînement.

In [ ]:
df = D.stratified_sample(df)
train, val, test = D.split(df, seed=C.SEED)
feat = D.select_features(train, log)
display(pd.DataFrame(log[-3:], columns=["step", "value"]))

sizes = pd.DataFrame({s: {"rows": len(d), "attacks": int(d.y.sum()), "attack_rate": d.y.mean()}
                      for s, d in [("train", train), ("validation", val), ("test", test)]}).T
display(sizes)
D.leakage_check(train, val, test, feat)

In [ ]:
Xtr, ytr = train[feat], train["y"]
Xva, yva = val[feat], val["y"]
Xte, yte = test[feat], test["y"]
SUSPECTS = D.resolve_suspects(feat)
print("features:", len(feat), "| suspects to test later:", SUSPECTS)

## 3. Modèles

| Modèle | Rôle | Déséquilibre | Hyperparamètres réglés |
|---|---|---|---|
| Régression logistique (+ standardisation) | baseline | `class_weight="balanced"` | `C` |
| Forêt aléatoire, 200 arbres | modèle expliqué | `balanced_subsample` | `max_depth`, `min_samples_leaf` |
| XGBoost, 300 arbres | comparaison | `scale_pos_weight` = négatifs / positifs | `max_depth`, `learning_rate` |

Réglage par validation croisée stratifiée à 5 plis **sur l'entraînement uniquement**, score = PR-AUC (`average_precision`). La recherche tourne sur un sous-ensemble stratifié de l'entraînement (`TUNE_MAX_ROWS`), puis le meilleur réglage est ré-entraîné sur tout l'entraînement. Le standardiseur fait partie du pipeline : il n'apprend que sur les plis d'entraînement.

Le **seuil d'alerte** est choisi sur la **validation** (F1 maximal). Le test ne sert à rien choisir.

In [ ]:
models, infos, p_val, thresholds = {}, [], {}, {}
for name, (est, grid) in M.make_models(ytr).items():
    model, info = M.tune_and_fit(name, est, grid, Xtr, ytr, train["family"])
    p, us = M.predict_time(model, Xva)
    thr = E.best_threshold(yva, p)
    info.update({"val_pr_auc": E.average_precision_score(yva, p), "val_threshold": thr, "inference_us_per_flow": us})
    models[name], p_val[name], thresholds[name] = model, p, thr
    infos.append(info)
    print(f"{name:18s} done  CV PR-AUC {info['cv_pr_auc']:.4f}")
pd.DataFrame(infos).set_index("model")

In [ ]:
display(Markdown("**Validation (sert au choix du modèle et du seuil)**"))
pd.DataFrame({n: E.scores(yva, p_val[n], thresholds[n]) for n in models}).T[
    ["threshold", "precision", "recall", "f1", "pr_auc", "FP", "FN"]]

## 4. Évaluation finale sur le test (utilisé une seule fois)

Les trois modèles sont gelés (paramètres + seuil). On les évalue maintenant sur le test.

In [ ]:
p_test = {n: m.predict_proba(Xte)[:, 1] for n, m in models.items()}
results = pd.DataFrame({n: E.scores(yte, p_test[n], thresholds[n]) for n in models}).T
results[["precision", "recall", "f1", "pr_auc", "roc_auc", "precision_benign", "recall_benign",
         "TP", "FP", "FN", "TN", "alerts_per_10k_flows", "false_alerts_per_10k_flows"]]

In [ ]:
E.plot_pr({n: (yte, p_test[n]) for n in models}, yte.mean(), "Precision-recall on test",
          C.FIG_DIR / "pr_test.png"); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, n in zip(axes, models):
    cm = E.confusion_matrix(yte, (p_test[n] >= thresholds[n]).astype(int), labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v:,}", ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["benign", "attack"]); ax.set_yticks([0, 1], ["benign", "attack"])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(n, fontsize=10)
fig.tight_layout(); fig.savefig(C.FIG_DIR / "confusion_test.png", dpi=150); plt.show()

### Coût des faux positifs et des faux négatifs (point de vue SOC)

- **Faux négatif** = une intrusion non détectée : l'attaquant continue (exfiltration, mouvement latéral). Coût élevé et invisible.
- **Faux positif** = une alerte à trier pour rien : du temps d'analyste et de la *fatigue d'alerte*, qui finit par faire ignorer de vraies alertes.

La ligne `false_alerts_per_10k_flows` traduit le seuil en charge de travail. *À compléter après exécution :* pour un réseau de N flux par jour, le modèle retenu produit environ … fausses alertes par jour et manque … attaques sur … ; nous jugeons ce compromis … parce que …

In [ ]:
best_name = max(models, key=lambda n: E.average_precision_score(yva, p_val[n]))   # chosen on VALIDATION
print("best model on validation:", best_name)
E.recall_by_family(test["family"], yte, p_test[best_name], thresholds[best_name])

## 5. Tâche 2 — trois mesures d'importance globale (forêt aléatoire)

| Mesure | Calculée sur | Ce qu'elle mesure | Biais connu |
|---|---|---|---|
| Importance interne (impureté) | entraînement | baisse moyenne d'impureté des nœuds qui utilisent la variable | favorise les variables continues à beaucoup de valeurs ; reflète l'entraînement |
| Importance par permutation | test (sous-échantillon stratifié) | perte de PR-AUC quand on mélange la variable | sous-estime les variables corrélées (l'autre prend le relais) |
| Moyenne des \|SHAP\| | test (≤ 2 000 flux) | contribution moyenne à la prédiction de chaque flux | partage le crédit entre variables corrélées |

In [ ]:
rf = models["RandomForest"]
alerts = X.pick_alerts(yte, p_test["RandomForest"], thresholds["RandomForest"])
shap_rows = X.shap_sample_rows(yte, [i for _, i in alerts])
explainer = X.shap_explainer(rf)
sv, base_value = X.shap_values_pos(explainer, Xte.iloc[shap_rows])
print(f"SHAP on {len(shap_rows):,} test flows ({int(yte.iloc[shap_rows].sum()):,} attacks), base value {base_value:.3f}")

imp = X.global_importances(rf, Xte, yte, np.abs(sv).mean(axis=0), feat)
imp.to_csv(C.FIG_DIR / "global_importances.csv")
imp.head(C.TOP_K)[["impurity", "rank_impurity", "permutation_test", "rank_permutation_test",
                   "mean_abs_shap", "rank_mean_abs_shap"]]

In [ ]:
X.plot_importances(imp, path=C.FIG_DIR / "importances_top15.png"); plt.show()
display(Markdown("**Accord entre les classements**"))
display(X.rank_agreement(imp))
display(Markdown("**Variables très corrélées dans le top 15 (|r| ≥ 0,95) — leurs importances se partagent**"))
X.correlated_pairs(Xtr, list(imp.head(C.TOP_K).index))

**Discussion (à compléter après exécution) :** quelles variables sont en tête dans les trois mesures ? Lesquelles changent de rang, et pourquoi (corrélation, variables continues favorisées par l'impureté, différence entraînement / test) ?

## 6. Tâche 3 — cinq alertes expliquées

Deux vrais positifs (le plus sûr et le plus limite), deux faux positifs (les plus sûrs) et un faux négatif (l'attaque la plus « ratée »). Pour chacune : graphique SHAP en cascade, puis la note qu'écrirait un analyste SOC. Le brouillon automatique liste les variables qui poussent vers « attaque » et vers « bénin » ; la note finale est rédigée à la main.

In [ ]:
pos = {r: k for k, r in enumerate(shap_rows)}
notes = {}
for kind, i in alerts:
    k = pos[i]
    title = f"{kind} — {test['family'].iloc[i]} — score {p_test['RandomForest'][i]:.3f}"
    X.plot_waterfall(sv[k], base_value, Xte.iloc[i].values, feat, title,
                     C.FIG_DIR / f"alert_{kind.split()[0]}_{i}.png"); plt.show()
    tab = X.alert_table(Xte.iloc[i].values, sv[k], feat)
    notes[kind] = X.analyst_note_draft(kind, test["family"].iloc[i], p_test["RandomForest"][i],
                                       thresholds["RandomForest"], tab)
    print(notes[kind], "\n")

**Notes d'analyste (à rédiger après exécution)** — modèle : *pourquoi l'alerte s'est déclenchée → verdict → action suivante*.

1. **VP, sûr :** …
2. **VP, limite :** …
3. **FP n° 1 :** …
4. **FP n° 2 :** …
5. **FN :** …

## 7. Tâche 4 — audit de plausibilité du top 10

Pour chaque variable : ce qu'elle mesure côté réseau, et si un vrai détecteur pourrait s'y fier. Une variable est **suspecte** si elle dépend du banc d'essai (systèmes, configuration, outil de capture) plutôt que du comportement de l'attaque.

In [ ]:
HINTS = {
    "destination_port": "nomme le service visé ; le modèle peut apprendre « port 22 = force brute » → SUSPECT",
    "init_win_bytes_forward": "fenêtre TCP initiale, fixée par l'OS de la machine qui ouvre la connexion → SUSPECT (banc d'essai)",
    "init_win_bytes_backward": "fenêtre TCP initiale côté serveur, dépend de l'OS de la victime → SUSPECT (banc d'essai)",
    "flow_duration": "durée du flux ; sensible aux appendices TCP de CICFlowMeter (Engelen 2021) → à vérifier",
    "fwd_iat_min": "écart minimal entre paquets aller ; rythme de l'outil d'attaque → plausible",
    "flow_iat_mean": "rythme moyen des paquets ; scans et DoS sont très réguliers → plausible",
    "bwd_packet_length_mean": "taille des réponses du serveur ; pages HTTP vs refus → plausible",
    "fwd_packet_length_mean": "taille moyenne des requêtes → plausible",
    "average_packet_size": "taille moyenne des paquets → plausible",
    "packet_length_mean": "taille moyenne des paquets → plausible",
    "total_length_of_fwd_packets": "volume envoyé par le client → plausible",
    "total_fwd_packets": "nombre de paquets aller → plausible",
    "total_backward_packets": "nombre de paquets retour ; un scan reçoit peu de réponses → plausible",
    "syn_flag_count": "drapeaux SYN ; signature de scan / SYN flood → plausible",
    "psh_flag_count": "drapeaux PSH ; données applicatives → plausible",
    "fwd_header_length": "taille cumulée des en-têtes aller ; dépend du nombre de paquets → plausible, redondante",
    "subflow_fwd_bytes": "copie quasi exacte d'autres variables de volume → redondante",
    "flow_bytes_s": "débit ; calcul instable pour les flux très courts (valeurs infinies) → à vérifier",
    "flow_packets_s": "débit en paquets ; instable pour les flux très courts → à vérifier",
    "act_data_pkt_fwd": "paquets aller avec données ; force brute vs scan → plausible",
    "min_seg_size_forward": "taille minimale d'en-tête TCP ; dépend des options TCP de l'OS → SUSPECT",
}
audit = imp.head(10)[["rank_mean_abs_shap", "rank_permutation_test", "rank_impurity"]].copy()
audit["what it measures / verdict (hint, to confirm)"] = [HINTS.get(f, "… à compléter") for f in audit.index]
audit["suspect"] = audit.index.isin(SUSPECTS)
audit

**Conclusion de l'audit (à compléter) :** variables suspectes retenues = … Justification réseau : …
Si l'audit ajoute d'autres suspects, les ajouter dans `SUSPECTS` ci-dessous avant la section 8.

## 8. Tâche 5 — ablation : ré-entraîner sans les variables suspectes

Même forêt, mêmes hyperparamètres, mêmes données — seules les variables suspectes sont retirées. Le nouveau seuil est choisi sur la validation. Si les performances s'effondrent, le modèle initial reposait sur des artefacts ; si elles tiennent, il apprend surtout du comportement.

In [ ]:
# SUSPECTS = SUSPECTS + ["..."]   # <- add suspects found in the audit
feat_abl = [f for f in feat if f not in SUSPECTS]
rf_abl = X.clone(rf).fit(Xtr[feat_abl], ytr)
p_va_abl = rf_abl.predict_proba(Xva[feat_abl])[:, 1]
thr_abl = E.best_threshold(yva, p_va_abl)
p_te_abl = rf_abl.predict_proba(Xte[feat_abl])[:, 1]

abl = pd.DataFrame({
    f"RF, all {len(feat)} features": E.scores(yte, p_test["RandomForest"], thresholds["RandomForest"]),
    f"RF without {len(SUSPECTS)} suspects": E.scores(yte, p_te_abl, thr_abl),
}).T[["precision", "recall", "f1", "pr_auc", "FP", "FN"]]
abl.loc["change"] = abl.iloc[1] - abl.iloc[0]
print("removed:", SUSPECTS)
abl

In [ ]:
fam_abl = pd.concat({
    "recall, all features": E.recall_by_family(test["family"], yte, p_test["RandomForest"], thresholds["RandomForest"])["recall"],
    "recall, without suspects": E.recall_by_family(test["family"], yte, p_te_abl, thr_abl)["recall"],
}, axis=1)
fam_abl["change"] = fam_abl.iloc[:, 1] - fam_abl.iloc[:, 0]
fam_abl.sort_values("change")

**Lecture (à compléter) :** quelles familles perdent le plus quand on retire les suspects ? (Exemple attendu : les forces brutes SSH/FTP si le modèle s'appuyait sur le port.) Le modèle initial était-il robuste ?

## 9. Tâche 6 — stabilité des explications

Pour chaque graine : nouveau découpage 60/20/20 **et** nouvelle forêt (mêmes hyperparamètres), puis moyenne des |SHAP| sur 1 000 flux de test. On compare les classements à la graine de référence : nombre de variables communes dans le top 10 et tau de Kendall sur toutes les variables.

In [ ]:
runs = {}
for s in C.STABILITY_SEEDS:
    tr_s, _, te_s = D.split(df, seed=s)
    runs[f"seed {s}"] = X.stability_run(rf, tr_s, te_s, feat, seed=s)
stab = X.stability_table(runs)
stab

In [ ]:
top10 = pd.DataFrame({n: s.rank(ascending=False).astype(int) for n, s in runs.items()})
top10.loc[runs[f"seed {C.STABILITY_SEEDS[0]}"].nlargest(10).index]

## 10. Extension — forêt aléatoire vs XGBoost

In [ ]:
sv_xgb, _ = X.shap_values_pos(X.shap_explainer(models["XGBoost"]), Xte.iloc[shap_rows])
cmp = pd.DataFrame({"RF mean|SHAP|": np.abs(sv).mean(0), "XGB mean|SHAP| (log-odds)": np.abs(sv_xgb).mean(0)}, index=feat)
cmp["rank RF"] = cmp.iloc[:, 0].rank(ascending=False).astype(int)
cmp["rank XGB"] = cmp.iloc[:, 1].rank(ascending=False).astype(int)
print("shared top-10:", len(set(cmp.nsmallest(10, "rank RF").index) & set(cmp.nsmallest(10, "rank XGB").index)))
cmp.sort_values("rank RF").head(10)

## 11. Conclusion et limites (à rédiger)

- **Réponse à la question centrale :** le modèle apprend-il des comportements d'attaque ou des particularités du jeu ? Appuyer sur l'ablation (section 8) et l'audit (section 7).
- **Ce que l'analyste SOC peut utiliser :** …
- **Limites :** échantillon de CIC-IDS2017 (pas tout le jeu) ; jeu de laboratoire de 2017 ; étiquettes imparfaites (Engelen 2021) ; découpage aléatoire et non temporel ; SHAP sur 2 000 flux ; corrélations entre variables qui brouillent l'attribution.
- **Pistes :** version corrigée du jeu, découpage par session, explications par famille d'attaque.

### Références
- G. Engelen, V. Rimmer, W. Joosen, « Troubleshooting an Intrusion Detection Dataset: the CICIDS2017 Case Study », IEEE Security and Privacy Workshops, 2021. doi:10.1109/SPW53761.2021.00009
- I. Sharafaldin, A. H. Lashkari, A. A. Ghorbani, « Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization », ICISSP 2018.
- S. M. Lundberg, S.-I. Lee, « A Unified Approach to Interpreting Model Predictions », NeurIPS 2017.